# Monitoring & Cost Optimization

## Model Monitor for Data Drift

Model Monitor detects data drift and model performance degradation. It compares production data against a baseline to identify distribution shifts.

In [ ]:
from sagemaker.model_monitor import DataQualityMonitor, DataCaptureConfig
import sagemaker

session = sagemaker.Session()
role = 'arn:aws:iam::123456789012:role/SageMakerRole'
bucket = session.default_bucket()

# Enable data capture on endpoint
data_capture_config = DataCaptureConfig(
    enabled=True,
    sampling_percentage=100,
    destination_s3_uri=f's3://{bucket}/data-capture/'
)

# Deploy endpoint with data capture
predictor = estimator.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large',
    data_capture_config=data_capture_config,
    endpoint_name='monitored-endpoint'
)

# Create baseline
monitor = DataQualityMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    sagemaker_session=session
)

# Create baseline from training data
monitor.suggest_baseline(
    baseline_dataset=f's3://{bucket}/train-data/data.csv',
    dataset_format='text/csv'
)

# Schedule monitoring
monitor.create_monitoring_schedule(
    monitor_schedule_name='data-quality-monitor',
    endpoint_input=f's3://{bucket}/data-capture/',
    output_s3_uri=f's3://{bucket}/monitoring-output/',
    statistics=monitor.baseline_statistics(),
    constraints=monitor.baseline_constraints(),
    schedule_expression='cron(0 * * * ? *)'  # Hourly
)

## Detecting Data Drift

In [ ]:
import boto3

sm_client = boto3.client('sagemaker')

# Get monitoring execution details
response = sm_client.list_monitoring_executions(
    MonitoringScheduleName='data-quality-monitor'
)

for execution in response['MonitoringExecutionSummaries']:
    print(f"Execution: {execution['MonitoringExecutionArn']}")
    print(f"Status: {execution['MonitoringExecutionStatus']}")
    
    # Get violations
    violations = sm_client.get_monitoring_schedule(
        MonitoringScheduleName='data-quality-monitor'
    )

## Endpoint Auto-Scaling

In [ ]:
import boto3

autoscaling = boto3.client('application-autoscaling')

# Register endpoint for auto-scaling
autoscaling.register_scalable_target(
    ServiceNamespace='sagemaker',
    ResourceId='endpoint/my-endpoint/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    MinCapacity=1,
    MaxCapacity=10
)

# Create scaling policy
autoscaling.put_scaling_policy(
    PolicyName='endpoint-scaling-policy',
    ServiceNamespace='sagemaker',
    ResourceId='endpoint/my-endpoint/variant/AllTraffic',
    ScalableDimension='sagemaker:variant:DesiredInstanceCount',
    PolicyType='TargetTrackingScaling',
    TargetTrackingScalingPolicyConfiguration={
        'TargetValue': 70.0,
        'PredefinedMetricSpecification': {
            'PredefinedMetricType': 'SageMakerVariantInvocationsPerInstance'
        },
        'ScaleOutCooldown': 300,
        'ScaleInCooldown': 300
    }
)

## Cost Optimization Strategies

In [ ]:
from sagemaker.estimator import Estimator

# Use spot instances for training
estimator = Estimator(
    image_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    output_path=f's3://{bucket}/training-output',
    sagemaker_session=session,
    use_spot_instances=True,
    max_run=3600,
    max_wait=5400
)

# Use serverless endpoints for variable traffic
from sagemaker.serverless import ServerlessInferenceConfig

serverless_config = ServerlessInferenceConfig(
    memory_size_in_mb=1024,
    max_concurrency=10
)

predictor = estimator.deploy(
    serverless_inference_config=serverless_config,
    endpoint_name='cost-optimized-endpoint'
)

# Use multi-model endpoints
from sagemaker.multidatamodel import MultiDataModel

multi_model = MultiDataModel(
    name='multi-model-endpoint',
    model_data_prefix=f's3://{bucket}/models/',
    model_name='xgboost-multi',
    container_uri='382416733822.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:latest',
    role=role,
    sagemaker_session=session
)

predictor = multi_model.deploy(
    initial_instance_count=1,
    instance_type='ml.m5.large'
)

## Monitoring Configuration

```json
{
  "monitoring_config": {
    "monitoring_schedule_name": "data-quality-monitor",
    "monitoring_job_definition": {
      "baseline_config": {
        "baselining_job_name": "baseline-job"
      },
      "monitoring_inputs": [
        {
          "endpoint_input": {
            "endpoint_name": "my-endpoint",
            "local_path": "/opt/ml/processing/input",
            "s3_input_mode": "File",
            "s3_data_distribution_type": "FullyReplicated"
          }
        }
      ],
      "monitoring_output_config": {
        "monitoring_outputs": [
          {
            "s3_output": {
              "s3_uri": "s3://my-bucket/monitoring-output/",
              "local_path": "/opt/ml/processing/output",
              "s3_upload_mode": "EndOfJob"
            }
          }
        ]
      },
      "monitoring_resources": {
        "cluster_config": {
          "instance_count": 1,
          "instance_type": "ml.m5.xlarge",
          "volume_size_in_gb": 30
        }
      }
    },
    "schedule_expression": "cron(0 * * * ? *)"
  }
}
```

## CloudWatch Metrics

In [ ]:
import boto3

cloudwatch = boto3.client('cloudwatch')

# Get endpoint invocation metrics
response = cloudwatch.get_metric_statistics(
    Namespace='AWS/SageMaker',
    MetricName='InvocationsPerInstance',
    Dimensions=[
        {
            'Name': 'EndpointName',
            'Value': 'my-endpoint'
        },
        {
            'Name': 'VariantName',
            'Value': 'AllTraffic'
        }
    ],
    StartTime='2024-01-01T00:00:00Z',
    EndTime='2024-01-02T00:00:00Z',
    Period=3600,
    Statistics=['Average', 'Sum']
)

print(f"Metrics: {response['Datapoints']}")

## References

### AWS SageMaker Documentation
- [SageMaker Developer Guide](https://docs.aws.amazon.com/sagemaker/latest/dg/)
- [SageMaker Python SDK](https://sagemaker.readthedocs.io/)
- [SageMaker Examples](https://github.com/aws/amazon-sagemaker-examples)

### Key Services
- [Model Monitor](https://docs.aws.amazon.com/sagemaker/latest/dg/model-monitor.html)
- [Feature Store](https://docs.aws.amazon.com/sagemaker/latest/dg/feature-store.html)
- [Pipelines](https://docs.aws.amazon.com/sagemaker/latest/dg/pipelines.html)
- [Autopilot](https://docs.aws.amazon.com/sagemaker/latest/dg/autopilot-automate-model-development.html)

### Best Practices
- Use spot instances for cost savings (up to 90%)
- Enable data capture for monitoring
- Implement auto-scaling for variable traffic
- Use multi-model endpoints for multiple models
- Monitor data drift regularly
- Version models in the model registry
- Automate workflows with pipelines

### Cost Optimization Tips
1. Use serverless endpoints for unpredictable traffic
2. Implement auto-scaling for consistent performance
3. Use spot training instances
4. Consolidate models on multi-model endpoints
5. Monitor and clean up unused resources
6. Use reserved instances for predictable workloads

## Quiz 1

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does Model Monitor detect?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="0">
      <span>Data drift and model performance degradation</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="1">
      <span>Training errors</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="2">
      <span>Deployment failures</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q8374920" value="3">
      <span>Cost overruns</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 2

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ What is data capture used for?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="0">
      <span>Training models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="1">
      <span>Recording predictions for monitoring</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="2">
      <span>Deploying endpoints</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q5729384" value="3">
      <span>Tuning hyperparameters</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 3

<div class="quiz" data-correct="0">
  <p class="font-semibold mb-3">❓ What does endpoint auto-scaling do?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="0">
      <span>Adjusts instance count based on traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="1">
      <span>Trains new models</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="2">
      <span>Monitors data drift</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q3847291" value="3">
      <span>Deploys models</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 4

<div class="quiz" data-correct="2">
  <p class="font-semibold mb-3">❓ Which strategy saves the most on training costs?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="0">
      <span>Using GPU instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="1">
      <span>Using on-demand instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="2">
      <span>Using spot instances</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q7291847" value="3">
      <span>Using reserved instances</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>

## Quiz 5

<div class="quiz" data-correct="1">
  <p class="font-semibold mb-3">❓ When should you use serverless endpoints?</p>
  <div class="space-y-2">
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="0">
      <span>For high-throughput consistent traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="1">
      <span>For variable or unpredictable traffic</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="2">
      <span>For batch processing</span>
    </label>
    <label class="flex items-center gap-2 cursor-pointer">
      <input type="radio" name="q9284756" value="3">
      <span>For training jobs</span>
    </label>
  </div>
  <button class="quiz-btn mt-3 px-4 py-2 bg-blue-600 text-white rounded text-sm font-medium hover:bg-blue-700">Check Answer</button>
  <p class="quiz-result text-sm mt-2 hidden"></p>
</div>